In [1]:

CLINICAL_RULES = [
    # ── 结构性规则 ──────────────────────────────────────────
    {
        "id":          "rule_cdr_high",
        "label":       "ClinicalRule",
        "name":        "CDR high risk threshold",
        "biomarker":   "cup_to_disc_ratio",
        "condition":   ">=",
        "threshold":   "0.7",
        "evidence":    "glaucoma",
        "strength":    "strong",
        "source":      "clinical_prior",
        "description": "CDR >= 0.7 is a well-established indicator of glaucomatous optic nerve damage",
    },
    {
        "id":          "rule_cdr_moderate",
        "label":       "ClinicalRule",
        "name":        "CDR moderate risk threshold",
        "biomarker":   "cup_to_disc_ratio",
        "condition":   ">=",
        "threshold":   "0.6",
        "evidence":    "moderate_risk",
        "strength":    "moderate",
        "source":      "clinical_prior",
        "description": "CDR 0.6-0.69 warrants monitoring for glaucoma progression",
    },
    # ── NeuralRim 规则 ──────────────────────────────────────
    {
        "id":          "rule_rim_thinning",
        "label":       "ClinicalRule",
        "name":        "Rim thinning rule",
        "biomarker":   "rim_thinning",
        "condition":   "==",
        "threshold":   "True",
        "evidence":    "glaucoma",
        "strength":    "strong",
        "source":      "data_derived",
        "description": "Rim thinning present in 100% of glaucoma cases and 0% of normal cases in this dataset",
    },
    {
        "id":          "rule_isnt",
        "label":       "ClinicalRule",
        "name":        "ISNT rule violation",
        "biomarker":   "isnt_rule_followed",
        "condition":   "==",
        "threshold":   "False",
        "evidence":    "glaucoma",
        "strength":    "strong",
        "source":      "data_derived",
        "description": "ISNT rule (Inferior > Superior > Nasal > Temporal rim) violated in 100% of glaucoma cases",
    },
    {
        "id":          "rule_rim_pallor",
        "label":       "ClinicalRule",
        "name":        "Rim pallor rule",
        "biomarker":   "rim_pallor",
        "condition":   "==",
        "threshold":   "True",
        "evidence":    "glaucoma",
        "strength":    "moderate",
        "source":      "data_derived",
        "description": "Rim pallor present in ~72% of glaucoma cases; absent in normal cases",
    },
    # ── Pathology 规则 ──────────────────────────────────────
    {
        "id":          "rule_bayoneting_notching",
        "label":       "ClinicalRule",
        "name":        "Bayoneting + notching co-occurrence",
        "biomarker":   "bayoneting,notching",
        "condition":   "both_true",
        "threshold":   "True",
        "evidence":    "glaucoma",
        "strength":    "strong",
        "source":      "data_derived",
        "description": "Bayoneting and notching co-occur with correlation 0.81; joint presence strongly indicates glaucoma",
    },
    {
        "id":          "rule_bayoneting",
        "label":       "ClinicalRule",
        "name":        "Bayoneting sign",
        "biomarker":   "bayoneting",
        "condition":   "==",
        "threshold":   "True",
        "evidence":    "glaucoma",
        "strength":    "moderate",
        "source":      "data_derived",
        "description": "Bayoneting present in ~73% of glaucoma cases; absent in normal",
    },
    {
        "id":          "rule_notching",
        "label":       "ClinicalRule",
        "name":        "Notching sign",
        "biomarker":   "notching",
        "condition":   "==",
        "threshold":   "True",
        "evidence":    "glaucoma",
        "strength":    "moderate",
        "source":      "data_derived",
        "description": "Notching present in ~73% of glaucoma cases; absent in normal",
    },
    {
        "id":          "rule_lds",
        "label":       "ClinicalRule",
        "name":        "Laminar dot sign",
        "biomarker":   "laminar_dot_sign",
        "condition":   "==",
        "threshold":   "True",
        "evidence":    "glaucoma",
        "strength":    "weak",
        "source":      "data_derived",
        "description": "Laminar dot sign present in ~55% of glaucoma cases; weakest single biomarker",
    },
    # ── 正常规则 ────────────────────────────────────────────
    {
        "id":          "rule_cdr_normal",
        "label":       "ClinicalRule",
        "name":        "CDR normal range",
        "biomarker":   "cup_to_disc_ratio",
        "condition":   "<=",
        "threshold":   "0.5",
        "evidence":    "normal",
        "strength":    "strong",
        "source":      "clinical_prior",
        "description": "CDR <= 0.5 is within normal range; normal cases concentrate at 0.4 in this dataset",
    },
    {
        "id":          "rule_isnt_normal",
        "label":       "ClinicalRule",
        "name":        "ISNT rule intact",
        "biomarker":   "isnt_rule_followed",
        "condition":   "==",
        "threshold":   "True",
        "evidence":    "normal",
        "strength":    "strong",
        "source":      "data_derived",
        "description": "ISNT rule followed in 100% of normal cases in this dataset",
    },
]

# 先要定义规则，这些经验来自于临床

In [2]:
import pandas as pd
from pathlib import Path
 
# 又要导入包

In [ ]:
OUT_DIR = Path("kg_output")
 
nodes_df = pd.read_csv(OUT_DIR / "kg_nodes.csv")
edges_df = pd.read_csv(OUT_DIR / "kg_edges.csv")
 
rules_df = pd.DataFrame(CLINICAL_RULES)
 
# 追加到 nodes（避免重复）
existing_ids = set(nodes_df["id"].values)
new_rules    = rules_df[~rules_df["id"].isin(existing_ids)]
nodes_df     = pd.concat([nodes_df, new_rules], ignore_index=True)
 
print(f"Added {len(new_rules)} ClinicalRule nodes")
print(nodes_df["label"].value_counts())

# 先把规则加入知识图谱的节点

Added 11 ClinicalRule nodes
label
FundusImage     689
OpticDisc       689
NeuralRim       689
Pathology       689
Diagnosis       689
ClinicalRule     11
RiskLevel         4
Name: count, dtype: int64


In [5]:
BIOMARKER_TO_LABEL = {
    "cup_to_disc_ratio": "OpticDisc",
    "sharp_edge":        "OpticDisc",
    "rim_thinning":      "NeuralRim",
    "isnt_rule_followed":"NeuralRim",
    "rim_pallor":        "NeuralRim",
    "bayoneting":        "Pathology",
    "notching":          "Pathology",
    "laminar_dot_sign":  "Pathology",
}
 
# 获取各类型节点 id
label_ids = {
    label: set(nodes_df[nodes_df.label == label]["id"].values)
    for label in ["OpticDisc", "NeuralRim", "Pathology"]
}
 
new_edges = []
for rule in CLINICAL_RULES:
    rule_id    = rule["id"]
    biomarkers = [b.strip() for b in rule["biomarker"].split(",")]
 
    # 找到涉及的节点类型
    node_labels = set()
    for bm in biomarkers:
        if bm in BIOMARKER_TO_LABEL:
            node_labels.add(BIOMARKER_TO_LABEL[bm])
 
    # 为每个对应类型的节点添加 GOVERNED_BY 边
    for nl in node_labels:
        for nid in label_ids.get(nl, []):
            new_edges.append({
                "src":      nid,
                "rel":      "GOVERNED_BY",
                "dst":      rule_id,
                "strength": rule["strength"],
            })
 
new_edges_df = pd.DataFrame(new_edges)
edges_df     = pd.concat([edges_df, new_edges_df], ignore_index=True)
 
print(f"\nAdded {len(new_edges_df):,} GOVERNED_BY edges")
print(edges_df["rel"].value_counts())

# 把每个biomarker定义到临床规则里面
 


Added 7,579 GOVERNED_BY edges
rel
GOVERNED_BY           15158
SUPPORTS_DIAGNOSIS     1444
HAS_OPTIC_DISC          689
HAS_RIM                 689
HAS_PATHOLOGY           689
HAS_DIAGNOSIS           689
HAS_RISK                689
Name: count, dtype: int64


In [6]:
diag_nodes = nodes_df[nodes_df.label == "Diagnosis"].copy()
# img -> diag mapping via HAS_DIAGNOSIS
has_diag = edges_df[edges_df.rel == "HAS_DIAGNOSIS"][["src","dst"]].rename(
    columns={"src": "img_id", "dst": "diag_id"}
)
# img -> od / rim / path
has_od   = edges_df[edges_df.rel == "HAS_OPTIC_DISC"][["src","dst"]].rename(columns={"src":"img_id","dst":"od_id"})
has_rim  = edges_df[edges_df.rel == "HAS_RIM"][["src","dst"]].rename(columns={"src":"img_id","dst":"rim_id"})
has_path = edges_df[edges_df.rel == "HAS_PATHOLOGY"][["src","dst"]].rename(columns={"src":"img_id","dst":"path_id"})
 
master = (
    has_diag
    .merge(has_od,   on="img_id")
    .merge(has_rim,  on="img_id")
    .merge(has_path, on="img_id")
)
 
od_cols   = nodes_df[nodes_df.label=="OpticDisc"][["id","cup_to_disc_ratio"]].rename(columns={"id":"od_id"})
rim_cols  = nodes_df[nodes_df.label=="NeuralRim"][["id","isnt_rule_followed","rim_pallor","rim_thinning"]].rename(columns={"id":"rim_id"})
path_cols = nodes_df[nodes_df.label=="Pathology"][["id","bayoneting","notching","laminar_dot_sign"]].rename(columns={"id":"path_id"})
diag_cols = nodes_df[nodes_df.label=="Diagnosis"][["id","risk_assessment"]].rename(columns={"id":"diag_id"})
 
master = (
    master
    .merge(od_cols,   on="od_id")
    .merge(rim_cols,  on="rim_id")
    .merge(path_cols, on="path_id")
    .merge(diag_cols, on="diag_id")
)
 
# bool 转换
for col in ["isnt_rule_followed","rim_pallor","rim_thinning","bayoneting","notching","laminar_dot_sign"]:
    master[col] = master[col].map({True:True, False:False, "True":True, "False":False})
 
master["cup_to_disc_ratio"] = pd.to_numeric(master["cup_to_disc_ratio"], errors="coerce")
 
supports_edges = []
for _, row in master.iterrows():
    diag_id = row["diag_id"]
    cdr     = row["cup_to_disc_ratio"]
 
    if pd.notna(cdr):
        if cdr >= 0.7:
            supports_edges.append({"src": diag_id, "rel": "SUPPORTS_RULE", "dst": "rule_cdr_high"})
        elif cdr >= 0.6:
            supports_edges.append({"src": diag_id, "rel": "SUPPORTS_RULE", "dst": "rule_cdr_moderate"})
        else:
            supports_edges.append({"src": diag_id, "rel": "SUPPORTS_RULE", "dst": "rule_cdr_normal"})
 
    if row["rim_thinning"] is True:
        supports_edges.append({"src": diag_id, "rel": "SUPPORTS_RULE", "dst": "rule_rim_thinning"})
    if row["isnt_rule_followed"] is False:
        supports_edges.append({"src": diag_id, "rel": "SUPPORTS_RULE", "dst": "rule_isnt"})
    if row["rim_pallor"] is True:
        supports_edges.append({"src": diag_id, "rel": "SUPPORTS_RULE", "dst": "rule_rim_pallor"})
    if row["bayoneting"] is True and row["notching"] is True:
        supports_edges.append({"src": diag_id, "rel": "SUPPORTS_RULE", "dst": "rule_bayoneting_notching"})
    if row["bayoneting"] is True:
        supports_edges.append({"src": diag_id, "rel": "SUPPORTS_RULE", "dst": "rule_bayoneting"})
    if row["notching"] is True:
        supports_edges.append({"src": diag_id, "rel": "SUPPORTS_RULE", "dst": "rule_notching"})
    if row["laminar_dot_sign"] is True:
        supports_edges.append({"src": diag_id, "rel": "SUPPORTS_RULE", "dst": "rule_lds"})
 
supports_df = pd.DataFrame(supports_edges)
edges_df    = pd.concat([edges_df, supports_df], ignore_index=True)
 
print(f"Added {len(supports_df):,} SUPPORTS_RULE edges")
print(edges_df["rel"].value_counts())

# 把就诊节点连上去，和支持它的临床规则链接

Added 3,004 SUPPORTS_RULE edges
rel
GOVERNED_BY           15158
SUPPORTS_RULE          3004
SUPPORTS_DIAGNOSIS     1444
HAS_OPTIC_DISC          689
HAS_RIM                 689
HAS_PATHOLOGY           689
HAS_DIAGNOSIS           689
HAS_RISK                689
Name: count, dtype: int64


In [ ]:
nodes_df.to_csv(OUT_DIR / "kg_nodes.csv", index=False)
edges_df.to_csv(OUT_DIR / "kg_edges.csv", index=False)
 
print(f"\n完成。")
print(f"Nodes : {len(nodes_df):,}  ->  kg_output/kg_nodes.csv")
print(f"Edges : {len(edges_df):,}  ->  kg_output/kg_edges.csv")

# 保存更新的CSV


完成。
Nodes : 3,460  ->  kg_output/kg_nodes.csv
Edges : 23,051  ->  kg_output/kg_edges.csv


In [8]:
print("\n=== 示例：一个 high-risk 病例的 SUPPORTS_RULE 链 ===")
 
# 取第一个 high-risk diagnosis
hr_diag = nodes_df[
    (nodes_df.label == "Diagnosis") &
    (nodes_df.risk_assessment == "high risk")
].iloc[0]
 
hr_rules = edges_df[
    (edges_df.src == hr_diag["id"]) &
    (edges_df.rel == "SUPPORTS_RULE")
]["dst"].values
 
print(f"Diagnosis id  : {hr_diag['id']}")
print(f"Risk          : {hr_diag['risk_assessment']}")
print(f"Confidence    : {hr_diag['confidence']}")
print(f"\nSupported rules:")
rule_lookup = rules_df.set_index("id")
for rid in hr_rules:
    r = rule_lookup.loc[rid]
    print(f"  [{r['strength']:8s}]  {r['name']}")

# 验证：查看一个 high-risk 病例的完整推理链


=== 示例：一个 high-risk 病例的 SUPPORTS_RULE 链 ===
Diagnosis id  : diag_a5561a59
Risk          : high risk
Confidence    : 0.9

Supported rules:
  [strong  ]  CDR high risk threshold
  [strong  ]  Rim thinning rule
  [strong  ]  ISNT rule violation
  [moderate]  Rim pallor rule
  [strong  ]  Bayoneting + notching co-occurrence
  [moderate]  Bayoneting sign
  [moderate]  Notching sign
  [weak    ]  Laminar dot sign
